In [1]:
import random
import json
import pickle
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
import tensorflow as tf
import random
import pickle
import json
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model
from tensorflow.keras import regularizers

# Download required modules
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [2]:
import zipfile
data = zipfile.ZipFile("/content/intents.zip")
data.extractall()

In [3]:
intents = json.loads(open("/content/intents.json").read())
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()
# Initialize lists to store words, classes, and documents
words = []
classes = []
documents = []

# List of characters to ignore in the patterns
ignore_letters = ['?', '!', '.', ',']

# Extract patterns and tags from intents data
for intent in intents['intents']:
    for pattern in intent['patterns']:
        # Tokenize the pattern and store in words list
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)

        # Append the tokenized pattern and its tag to the documents list
        documents.append((word_list, intent['tag']))

        # Append the tag to classes list, if it's not already present
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-3-4e7a435d4451>", line 16, in <cell line: 13>
    word_list = nltk.word_tokenize(pattern)
  File "/usr/local/lib/python3.10/dist-packages/nltk/tokenize/__init__.py", line 130, in word_tokenize
    return [
  File "/usr/local/lib/python3.10/dist-packages/nltk/tokenize/__init__.py", line 131, in <listcomp>
    token for sent in sentences for token in _treebank_word_tokenizer.tokenize(sent)
  File "/usr/local/lib/python3.10/dist-packages/nltk/tokenize/destructive.py", line 160, in tokenize
    text = regexp.sub(substitution, text)
  File "/usr/lib/python3.10/re.py", line 324, in _subx
    def _subx(pattern, template):
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/di

TypeError: object of type 'NoneType' has no len()

In [ ]:
# Lemmatize words and remove duplicates
words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in ignore_letters]
words = sorted(list(set(words)))  # set() will remove duplicates automatically
classes = sorted(list(set(classes)))

In [ ]:
# Pickle the words and classes for future use
with open('words.pkl', 'wb') as f:
    pickle.dump(words, f)
with open('classes.pkl', 'wb') as f:
    pickle.dump(classes, f)


In [ ]:
# Initialize training data
training = []
output_empty = [0] * len(classes)

In [ ]:
# Prepare bag of words representation for each document
for document in documents:
    word_patterns = [lemmatizer.lemmatize(word.lower()) for word in document[0] if word not in ignore_letters]
    bag = [1 if word in word_patterns else 0 for word in words]

    output_row = list(output_empty)
    output_row[classes.index(document[1])] = 1

    training.append([bag, output_row])
    # Shuffle the training data
random.shuffle(training)

In [ ]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Load intents data from json file
intents = json.loads(open('data.json').read())

# Load pre-trained words and classes from pickle files
words = pickle.load(open('words.pkl', 'rb'))
classes = pickle.load(open('classes.pkl', 'rb'))



In [ ]:
def clean_up_sentences(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word) for word in sentence_words]
    return sentence_words


In [ ]:
# Convert training data to NumPy arrays
train_x = np.array([x[0] for x in training])
train_y = np.array([x[1] for x in training])

# Split the data into training and validation sets
from sklearn.model_selection import train_test_split
train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size=0.2, random_state=42)


In [ ]:
# Build the model with dropout and L2 regularization
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))


In [ ]:
# Compile the model with the optimizer, loss function, and evaluation metric
model.compile(loss='categorical_crossentropy', optimizer="adam", metrics=['accuracy'])

In [ ]:
# Define early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
# Fit the model to the training data with specified parameters
model.fit(np.array(train_x), np.array(train_y), epochs=100, batch_size = 5, verbose=1)


In [ ]:
model.save('chatbot_model.h5')

In [ ]:

# Evaluate the model on the validation data
val_loss, val_acc = model.evaluate(val_x, val_y)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_acc)


In [ ]:
model.save('chatbot_model.keras')

In [ ]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Load intents data from json file
intents = json.loads(open('data.json').read())

# Load pre-trained words and classes from pickle files
words = pickle.load(open('words.pkl', 'rb'))
classes = pickle.load(open('classes.pkl', 'rb'))

# Load pre trained model
model = tf.keras.models.load_model('chatbot_model.keras')


In [ ]:
def clean_up_sentences(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word) for word in sentence_words]
    return sentence_words


In [ ]:
# Function to create bag of words from sentence
def bag_of_words(sentence):
    sentence_words = clean_up_sentences(sentence)
    bag = [0] * len(words)
    for w in sentence_words:
        if w in words:
            index = words.index(w)
            bag[index] = 1
    return np.array(bag)


In [ ]:
# Function to predict the class of the input sentence
def predict_class(sentence):
    ERROR_THRESHOLD = 0.25
    bow = bag_of_words(sentence)
    res = model.predict(np.array([bow]))[0]
    results = [(i, r) for i, r in enumerate(res) if r > ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)
    return [{'intent': classes[r[0]], 'probability': str(r[1])} for r in results]

In [ ]:
# Function to get the response for the input sentence
def get_response(intents_list, intents_json):
    if intents_list:
        tag = intents_list[0]['intent']
        intents_dict = {i['tag']: i for i in intents_json['intents']}
    else:
        return "No intent found"

    try:
        result = random.choice(intents_dict[tag]['responses'])
    except KeyError:
        result = "I am sorry, I am not sure how to respond to that."

    return result

#def get_response(intents_list, intents_json):
    if intents_list:
        tag = intents_list[0]['intent']
        intents_dict = {i['tag']: i for i in intents_json['intents']}
    else:
        return "No intent found"

    try:
        result = random.choice(intents_dict[tag]['responses'])
    except KeyError:
        result = "I am sorry, I am not sure how to respond to that."

    return result

# Infinite loop to keep the chatbot running
while True:
    user_input = input("User: ")
    if user_input.lower() == 'exit':
        print("Chatbot: Goodbye!")
        break
    else:
        ints = predict_class(user_input)
        res = get_response(ints, intents)
        print("Chatbot:", res)

In [ ]:
# from tkinter import *
# import time
# import tkinter.messagebox

# saved_username = ["You"]
# window_size = "500x500"


# class ChatInterface(Frame):

#     def __init__(self, master=None):
#         Frame.__init__(self, master)
#         self.master = master
#         self.tl_bg = "#EEEEEE"
#         self.tl_bg2 = "#EEEEEE"
#         self.tl_fg = "#000000"
#         self.font = "Verdana 10"

#         menu = Menu(self.master)
#         self.master.config(menu=menu, bd=5)
#         # Menu bar

#         # File
#         file = Menu(menu, tearoff=0)
#         menu.add_cascade(label="File", menu=file)
#         # file.add_command(label="Save Chat Log", command=self.save_chat)
#         file.add_command(label="Clear Chat", command=self.clear_chat)
#         #  file.add_separator()
#         file.add_command(label="Exit", command=self.chatexit)

#         # Options
#         options = Menu(menu, tearoff=0)
#         menu.add_cascade(label="Options", menu=options)

#         # font
#         font = Menu(options, tearoff=0)
#         options.add_cascade(label="Font", menu=font)
#         font.add_command(label="Default", command=self.font_change_default)
#         font.add_command(label="Times", command=self.font_change_times)
#         font.add_command(label="System", command=self.font_change_system)
#         font.add_command(label="Helvetica", command=self.font_change_helvetica)
#         font.add_command(label="Fixedsys", command=self.font_change_fixedsys)

#         # color theme
#         color_theme = Menu(options, tearoff=0)
#         options.add_cascade(label="Color Theme", menu=color_theme)
#         color_theme.add_command(label="Default", command=self.color_theme_default)
#         # color_theme.add_command(label="Night",command=self.)
#         color_theme.add_command(label="Grey", command=self.color_theme_grey)
#         color_theme.add_command(label="Blue", command=self.color_theme_dark_blue)

#         color_theme.add_command(label="Torque", command=self.color_theme_turquoise)
#         color_theme.add_command(label="Hacker", command=self.color_theme_hacker)
#         # color_theme.add_command(label='Mkbhd',command=self.MKBHD)

#         help_option = Menu(menu, tearoff=0)
#         menu.add_cascade(label="Help", menu=help_option)
#         # help_option.add_command(label="Features", command=self.features_msg)
#         help_option.add_command(label="About MedBot", command=self.msg)
#         help_option.add_command(label="Developers", command=self.about)

#         self.text_frame = Frame(self.master, bd=6)
#         self.text_frame.pack(expand=True, fill=BOTH)

#         # scrollbar for text box
#         self.text_box_scrollbar = Scrollbar(self.text_frame, bd=0)
#         self.text_box_scrollbar.pack(fill=Y, side=RIGHT)

#         # contains messages
#         self.text_box = Text(self.text_frame, yscrollcommand=self.text_box_scrollbar.set, state=DISABLED,
#                              bd=1, padx=6, pady=6, spacing3=8, wrap=WORD, bg=None, font="Verdana 10", relief=GROOVE,
#                              width=10, height=1)
#         self.text_box.pack(expand=True, fill=BOTH)
#         self.text_box_scrollbar.config(command=self.text_box.yview)

#         # frame containing user entry field
#         self.entry_frame = Frame(self.master, bd=1)
#         self.entry_frame.pack(side=LEFT, fill=BOTH, expand=True)

#         # entry field
#         self.entry_field = Entry(self.entry_frame, bd=1, justify=LEFT)
#         self.entry_field.pack(fill=X, padx=6, pady=6, ipady=3)
#         # self.users_message = self.entry_field.get()

#         # frame containing send button and emoji button
#         self.send_button_frame = Frame(self.master, bd=0)
#         self.send_button_frame.pack(fill=BOTH)

#         # send button
#         self.send_button = Button(self.send_button_frame, text="Send", width=5, relief=GROOVE, bg='white',
#                                   bd=1, command=lambda: self.send_message_insert(None), activebackground="#FFFFFF",
#                                   activeforeground="#000000")
#         self.send_button.pack(side=LEFT, ipady=8, expand=True)
#         self.master.bind("<Return>", self.send_message_insert)
#         self.last_sent_label(date="No messages sent.")

#     def last_sent_label(self, date):

#         try:
#             self.sent_label.destroy()
#         except AttributeError:
#             pass

#         self.sent_label = Label(self.entry_frame, font="Verdana 7", text=date, bg=self.tl_bg2, fg=self.tl_fg)
#         self.sent_label.pack(side=LEFT, fill=BOTH, padx=3)

#     def clear_chat(self):
#         self.text_box.config(state=NORMAL)
#         self.last_sent_label(date="No messages sent.")
#         self.text_box.delete(1.0, END)
#         self.text_box.delete(1.0, END)
#         self.text_box.config(state=DISABLED)

#     def chatexit(self):
#         exit()

#     def msg(self):
#         tkinter.messagebox.showinfo("MedBot v1.0",
#                                     'MedBot is a chatbot for answering health related queries\nIt is based on retrival-based NLP using pythons NLTK tool-kit module\nGUI is based on Tkinter\nIt can answer questions regarding users health status')

#     def about(self):
#         tkinter.messagebox.showinfo("MedBot Developers",
#                                     "1.Samarth Kumar Pal\n2.Rakesh Kumar\n3.Amber Kakkar\n4.Akash Upadhyay")

#     def send_message_insert(self, message):
#         user_input = self.entry_field.get()
#         pr1 = "Human : " + user_input + "\n"
#         self.text_box.configure(state=NORMAL)
#         self.text_box.insert(END, pr1)
#         self.text_box.configure(state=DISABLED)
#         self.text_box.see(END)
#         ob = chat(user_input)
#         pr = "MedBot : " + ob + "\n"
#         self.text_box.configure(state=NORMAL)
#         self.text_box.insert(END, pr)
#         self.text_box.configure(state=DISABLED)
#         self.text_box.see(END)
#         self.last_sent_label(str(time.strftime("Last message sent: " + '%B %d, %Y' + ' at ' + '%I:%M %p')))
#         self.entry_field.delete(0, END)

#     def font_change_default(self):
#         self.text_box.config(font="Verdana 10")
#         self.entry_field.config(font="Verdana 10")
#         self.font = "Verdana 10"

#     def font_change_times(self):
#         self.text_box.config(font="Times")
#         self.entry_field.config(font="Times")
#         self.font = "Times"

#     def font_change_system(self):
#         self.text_box.config(font="System")
#         self.entry_field.config(font="System")
#         self.font = "System"

#     def font_change_helvetica(self):
#         self.text_box.config(font="helvetica 10")
#         self.entry_field.config(font="helvetica 10")
#         self.font = "helvetica 10"

#     def font_change_fixedsys(self):
#         self.text_box.config(font="fixedsys")
#         self.entry_field.config(font="fixedsys")
#         self.font = "fixedsys"

#     def color_theme_default(self):
#         self.master.config(bg="#EEEEEE")
#         self.text_frame.config(bg="#EEEEEE")
#         self.entry_frame.config(bg="#EEEEEE")
#         self.text_box.config(bg="#FFFFFF", fg="#000000")
#         self.entry_field.config(bg="#FFFFFF", fg="#000000", insertbackground="#000000")
#         self.send_button_frame.config(bg="#EEEEEE")
#         self.send_button.config(bg="#FFFFFF", fg="#000000", activebackground="#FFFFFF", activeforeground="#000000")
#         self.sent_label.config(bg="#EEEEEE", fg="#000000")

#         self.tl_bg = "#FFFFFF"
#         self.tl_bg2 = "#EEEEEE"
#         self.tl_fg = "#000000"

#     # Dark
#     def color_theme_dark(self):
#         self.master.config(bg="#2a2b2d")
#         self.text_frame.config(bg="#2a2b2d")
#         self.text_box.config(bg="#212121", fg="#FFFFFF")
#         self.entry_frame.config(bg="#2a2b2d")
#         self.entry_field.config(bg="#212121", fg="#FFFFFF", insertbackground="#FFFFFF")
#         self.send_button_frame.config(bg="#2a2b2d")
#         self.send_button.config(bg="#212121", fg="#FFFFFF", activebackground="#212121", activeforeground="#FFFFFF")
#         self.sent_label.config(bg="#2a2b2d", fg="#FFFFFF")

#         self.tl_bg = "#212121"
#         self.tl_bg2 = "#2a2b2d"
#         self.tl_fg = "#FFFFFF"

#     # Grey
#     def color_theme_grey(self):
#         self.master.config(bg="#444444")
#         self.text_frame.config(bg="#444444")
#         self.text_box.config(bg="#4f4f4f", fg="#ffffff")
#         self.entry_frame.config(bg="#444444")
#         self.entry_field.config(bg="#4f4f4f", fg="#ffffff", insertbackground="#ffffff")
#         self.send_button_frame.config(bg="#444444")
#         self.send_button.config(bg="#4f4f4f", fg="#ffffff", activebackground="#4f4f4f", activeforeground="#ffffff")
#         self.sent_label.config(bg="#444444", fg="#ffffff")

#         self.tl_bg = "#4f4f4f"
#         self.tl_bg2 = "#444444"
#         self.tl_fg = "#ffffff"

#     def color_theme_turquoise(self):
#         self.master.config(bg="#003333")
#         self.text_frame.config(bg="#003333")
#         self.text_box.config(bg="#669999", fg="#FFFFFF")
#         self.entry_frame.config(bg="#003333")
#         self.entry_field.config(bg="#669999", fg="#FFFFFF", insertbackground="#FFFFFF")
#         self.send_button_frame.config(bg="#003333")
#         self.send_button.config(bg="#669999", fg="#FFFFFF", activebackground="#669999", activeforeground="#FFFFFF")
#         self.sent_label.config(bg="#003333", fg="#FFFFFF")

#         self.tl_bg = "#669999"
#         self.tl_bg2 = "#003333"
#         self.tl_fg = "#FFFFFF"

#         # Blue

#     def color_theme_dark_blue(self):
#         self.master.config(bg="#263b54")
#         self.text_frame.config(bg="#263b54")
#         self.text_box.config(bg="#1c2e44", fg="#FFFFFF")
#         self.entry_frame.config(bg="#263b54")
#         self.entry_field.config(bg="#1c2e44", fg="#FFFFFF", insertbackground="#FFFFFF")
#         self.send_button_frame.config(bg="#263b54")
#         self.send_button.config(bg="#1c2e44", fg="#FFFFFF", activebackground="#1c2e44", activeforeground="#FFFFFF")
#         self.sent_label.config(bg="#263b54", fg="#FFFFFF")

#         self.tl_bg = "#1c2e44"
#         self.tl_bg2 = "#263b54"
#         self.tl_fg = "#FFFFFF"

#     # Torque
#     def color_theme_turquoise(self):
#         self.master.config(bg="#003333")
#         self.text_frame.config(bg="#003333")
#         self.text_box.config(bg="#669999", fg="#FFFFFF")
#         self.entry_frame.config(bg="#003333")
#         self.entry_field.config(bg="#669999", fg="#FFFFFF", insertbackground="#FFFFFF")
#         self.send_button_frame.config(bg="#003333")
#         self.send_button.config(bg="#669999", fg="#FFFFFF", activebackground="#669999", activeforeground="#FFFFFF")
#         self.sent_label.config(bg="#003333", fg="#FFFFFF")

#         self.tl_bg = "#669999"
#         self.tl_bg2 = "#003333"
#         self.tl_fg = "#FFFFFF"

#     # Hacker
#     def color_theme_hacker(self):
#         self.master.config(bg="#0F0F0F")
#         self.text_frame.config(bg="#0F0F0F")
#         self.entry_frame.config(bg="#0F0F0F")
#         self.text_box.config(bg="#0F0F0F", fg="#33FF33")
#         self.entry_field.config(bg="#0F0F0F", fg="#33FF33", insertbackground="#33FF33")
#         self.send_button_frame.config(bg="#0F0F0F")
#         self.send_button.config(bg="#0F0F0F", fg="#FFFFFF", activebackground="#0F0F0F", activeforeground="#FFFFFF")
#         self.sent_label.config(bg="#0F0F0F", fg="#33FF33")

#         self.tl_bg = "#0F0F0F"
#         self.tl_bg2 = "#0F0F0F"
#         self.tl_fg = "#33FF33"

#     # Default font and color theme
#     def default_format(self):
#         self.font_change_default()
#         self.color_theme_default()


# root = Tk()

# a = ChatInterface(root)
# root.geometry(window_size)
# root.title("MedBot")
# root.iconbitmap('MedBot.jpg')
# root.mainloop()

In [ ]:
# #while True:
#     message = input("")
#     ints = predict_class(message)
#     res = get_response(ints, intents)
#     print(res)

#     # Check if the user wants to exit
#     if message.lower() == 'exit':
#         print("Chatbot: Goodbye!")
#         break

In [ ]:
# import numpy as np
# import tensorflow as tf
# from tensorflow import keras
# from sklearn.model_selection import train_test_split
# import numpy as np
# import matplotlib.pyplot as plt
# import tensorflow_datasets as tfds
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.utils import to_categorical
# from tensorflow.keras.preprocessing.text import Tokenizer
# from tensorflow.keras.preprocessing.sequence import pad_sequences
# from tensorflow.keras.layers import LSTM, GRU, Flatten, Conv1D , Dense
# import numpy as np
# import os
# import csv
# from sklearn.preprocessing import LabelEncoder
# from keras.utils import to_categorical

# import json

# # Load the JSON file
# with open("/content/csvjson.json", 'r') as f:
#     datastore = json.load(f)

# # Initialize the lists
# sentences = []
# labels = []

# # Collect sentences and labels into the lists
# for item in datastore['intents']:
#     patterns = item['patterns']
#     tag = item['tag']
#     for pattern in patterns:
#         sentences.append(pattern)
#         labels.append(tag)



# training_size = 200

# # Split the sentences
# training_sentences = sentences[0:training_size]
# testing_sentences = sentences[training_size:]

# # Split the labels
# training_labels = labels[0:training_size]
# testing_labels = labels[training_size:]

# # Tokenize the text
# tokenizer = Tokenizer()
# tokenizer.fit_on_texts(sentences)
# word_index = tokenizer.word_index

# # Convert text to sequences
# sequences = tokenizer.texts_to_sequences(sentences)

# # Convert labels to categorical
# label_encoder = LabelEncoder()
# encoded_labels = label_encoder.fit_transform(labels)
# num_classes = len(label_encoder.classes_)
# categorical_labels = to_categorical(encoded_labels, num_classes=num_classes)

# # Pad sequences
# max_sequence_length = max([len(seq) for seq in sequences])
# padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

# # Split data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(padded_sequences, categorical_labels, test_size=0.2, random_state=42)


# # Build the BiLSTM model
# model = Sequential([
#     tf.keras.layers.Embedding(input_dim=len(word_index) + 1, output_dim=100, input_length=max_sequence_length),
#     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
#     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
#     tf.keras.layers.Dense(64, activation='relu'),
#     tf.keras.layers.Dense(num_classes, activation='softmax')
# ])

# # Print the model summary
# model.summary()


# # Compile the model
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# # Train the model
# history = model.fit(X_train, y_train, epochs=15, batch_size=32, validation_split=0.1)

# # Evaluate the model
# test_loss, test_acc = model.evaluate(X_test, y_test)
# print("Test Accuracy:", test_acc)

# def preprocess_text(text):
#     # Tokenize the text
#     sequence = tokenizer.texts_to_sequences([text])
#     # Pad sequence
#     padded_sequence = pad_sequences(sequence, maxlen=max_sequence_length, padding='post')
#     return padded_sequence

# # def predict_intent(text):
# #     preprocessed_text = preprocess_text(text)
# #     predictions = model.predict(preprocessed_text)
# #     predicted_label = label_encoder.inverse_transform([np.argmax(predictions)])
# #     return predicted_label[0]

# # datastore = {
# #     "intents": [
# #         {
# #             "tag": "Cuts",
# #             "patterns": ["What to do if Cuts?", "How to cure Cuts?", "Which medicine to apply for Cuts?", "what to apply on cuts?", "Cuts"],
# #             "responses": ["Wash the cut properly to prevent infection and stop the bleeding by applying pressure for 1-2 minutes until bleeding stops. Apply Petroleum Jelly to make sure that the wound is moist for quick healing. Finally, cover the cut with a sterile bandage. Pain relievers such as acetaminophen can be applied."],
# #             "context_set": ""
# #         },
# #         {
# #             "tag": "Abrasions",
# #             "patterns": [
# #                 "how do you treat abrasions?",
# #                 "Do Abrasions cause scars?",
# #                 "Abrasions",
# #                 "what to do if abrasions?",
# #                 "Which medicine to apply for abrasions?",
# #                 "How to cure abrasions?"
# #             ],
# #             "responses": ["Begin with washed hands. Gently clean the area with cool to lukewarm water and mild soap. Remove dirt or other particles from the wound using sterilized tweezers. For a mild scrape that’s not bleeding, leave the wound uncovered. If the wound is bleeding, use a clean cloth or bandage, and apply gentle pressure to the area to stop any bleeding. Cover a wound that bled with a thin layer of topical antibiotic ointment, like Bacitracin, or a sterile moisture barrier ointment, like Aquaphor. Cover it with a clean bandage or gauze. Gently clean the wound and change the ointment and bandage once per day. Watch the area for signs of infection, like pain or redness and swelling. See your doctor if you suspect infection."],
# #             "context_set": ""
# #         },
# #         {"tag": "stings",
# #          "patterns": ["How do you treat Sting?", "Stings", "What to do if you get a sting?", "Which medicine to apply if sting?"],
# #          "responses": ["Remove any stingers immediately. Some experts recommend scraping out the stinger with a credit card. Applying ice to the site may provide some mild relief. Apply ice for 20 minutes once every hour as needed. Wrap the ice in a towel or keep a cloth between the ice and skin to keep from freezing the skin. Taking an antihistamine such as diphenhydramine (Benadryl) or a nonsedating one such as loratadine (Claritin) will help with itching and swelling. Take acetaminophen (Tylenol) or ibuprofen (Motrin)for pain relief as needed. Wash the sting site with soap and water. Placing hydrocortisone cream on the sting can help relieve redness, itching, and swelling."],
# #          "context_set": ""
# #         },

# #         {"tag": "Splinter",
# #          "patterns": ["How to remove Splinters", "How to cure Splinters?", "What to do if I have splinters?", "How do you bring a splinter to the surface?"],
# #          "responses": ["1. SOAK IT IN EPSOM SALTS. Dissolve a cup of the salts into a warm bath and soak whatever part of the body has the splinter. Failing that, you can also put some of the salts onto a bandage pad and leave it covered for a day; this will eventually help bring the splinter to the surface. 2. VINEGAR OR OIL. Another simple way to draw out that stubborn splinter is to soak the affected area in oil (olive or corn) or white vinegar. Just pour some in a bowl and soak the area for around 20 to 30 minutes,"],
# #          "context_set": ""
# #         },

# #         {"tag": "Sprains",
# #          "patterns": ["How do you treat a sprain?", "what to do if i get a sprain?", "Which cream to apply if i get a sprain?", "Which medicine to apply if I get a sprain?"],
# #          "responses": ["Use an ice pack or ice slush bath immediately for 15 to 20 minutes and repeat every two to three hours while you're awake. To help stop swelling, compress the ankle with an elastic bandage until the swelling stops. In most cases, pain relievers — such as ibuprofen (Advil, Motrin IB, others) or naproxen sodium (Aleve, others) or acetaminophen (Tylenol, others) are enough to manage the pain of a sprained ankle."],
# #          "context_set": ""
# #         },

# #         {"tag": "Strains",
# #          "patterns": ["How do you treat a strain?", "what to do if i get a strain?", "Which cream to apply if i get a strain?", "Which medicine to apply if I get a strain?", "How do you diagnose a strain?", "Is heat or ice better for a pulled muscle?"],
# #          "responses": ["Rest,Ice,Compression and Elevation can be used to cure strains. Avoid using your muscle for a few days, especially if movement causes an increase in pain and also Apply ice immediately after injuring your muscle. This will minimize swelling. Don’t put ice directly on your skin. Use an ice pack or wrap ice in a towel. To reduce swelling, wrap the affected area with an elastic bandage until swelling comes down."],
# #          "context_set": ""
# #         },

# #         {"tag": "Fever",
# #          "patterns": ["How do you treat a mild Fever?", "what to do if i get a mild fever?", "Which medicine to take if I get a mild fever?", "fever"],
# #          "responses": ["To treat a fever at home: 1)Drink plenty of fluids to stay hydrated. 2)Dress in lightweight clothing. 3)Use a light blanket if you feel chilled, until the chills end. 4)Take acetaminophen (Tylenol, others) or ibuprofen (Advil, Motrin IB, others). 5) Get medical help if the fever lasts more than five days in a row or is higher than 103 F (39.4 C)"],
# #          "context_set": ""
# #         },

# #         {"tag": "Heatstroke",
# #          "patterns": ["How do you treat Heatstroke?", "What to do if I have a heatstroke?", "What to do if someone has a heatstroke?", "What to do if someone faints from heatstroke?"],
# #          "responses": ["Move the person out of the heat and into a shady or air-conditioned place. Lay the person down and elevate the legs and feet slightly. Remove tight or heavy clothing. Have the person drink cool water or fluids with electrolytes such as sports drinks. Cool the person by spraying or sponging with cool water and fanning. Monitor the person carefully."],
# #          "context_set": ""
# #         }
# #     ]
# # }


# # import string

# # def predict_intent(user_input):
# #     # Remove punctuation from user input
# #     user_input_cleaned = user_input.translate(str.maketrans('', '', string.punctuation))

# #     # Predict intent based on cleaned user input
# #     for intent in datastore['intents']:
# #         for pattern in intent['patterns']:
# #             # Remove punctuation from pattern for matching
# #             pattern_cleaned = pattern.translate(str.maketrans('', '', string.punctuation))
# #             if pattern_cleaned.lower() in user_input_cleaned.lower():
# #                 return intent['tag']
# #     return None


# import string
# import numpy as np
# from difflib import SequenceMatcher

# def calculate_similarity(str1, str2):
#     """
#     Calculate similarity between two strings using SequenceMatcher.
#     """
#     matcher = SequenceMatcher(None, str1, str2)
#     return matcher.ratio() * 100

# def predict_intent(user_input):
#     # Remove punctuation from user input
#     user_input_cleaned = user_input.translate(str.maketrans('', '', string.punctuation))

#     # Predict intent based on cleaned user input
#     for intent in datastore['intents']:
#         # Check if the user input exactly matches the intent tag or any pattern
#         if user_input_cleaned.lower() == intent['tag'].lower() or any(user_input_cleaned.lower() == pattern.lower() for pattern in intent['patterns']):
#             return intent['tag']

#         # Check similarity between user input and patterns
#         for pattern in intent['patterns']:
#             # Remove punctuation from pattern for matching
#             pattern_cleaned = pattern.translate(str.maketrans('', '', string.punctuation))
#             # Calculate similarity between user input and pattern
#             similarity = calculate_similarity(user_input_cleaned.lower(), pattern_cleaned.lower())
#             if similarity >= 40:  # Adjust the threshold as needed
#                 return intent['tag']
#     return None

# def chat():
#     # Initial message
#     print("Chatbot: Hi! How can I assist you today? Type 'exit' to end the conversation.")

#     while True:
#         # User input
#         user_input = input("You: ")

#         # Check if user wants to exit
#         if user_input.lower() == 'exit':
#             print("Chatbot: Goodbye!")
#             break

#         # Predict intent based on user input
#         intent = predict_intent(user_input)

#         # Get responses for the predicted intent
#         if intent:
#             responses = [item['responses'] for item in datastore['intents'] if item['tag'] == intent]
#             response = np.random.choice(responses[0]) if responses else "I'm not sure how to respond to that."
#             print("Chatbot:", response)
#         else:
#             # If no intent is found, inform the user
#             print("Chatbot: Sorry, I didn't understand that.")

# # def chat():
# #     # Initial message
# #     print("Chatbot: Hi! How can I assist you today? Type 'exit' to end the conversation.")

# #     while True:
# #         # User input
# #         user_input = input("You: ")

# #         # Check if user wants to exit
# #         if user_input.lower() == 'exit':
# #             print("Chatbot: Goodbye!")
# #             break

# #         # Predict intent based on user input
# #         intent = predict_intent(user_input)

# #         # Get responses for the predicted intent
# #         if intent:
# #             responses = [item['responses'] for item in datastore['intents'] if item['tag'] == intent]
# #             response = np.random.choice(responses[0]) if responses else "I'm not sure how to respond to that."
# #             print("Chatbot:", response)
# #         else:
# #             # If no intent is found, inform the user
# #             print("Chatbot: Sorry, I didn't understand that.")

# # Start the chat